# Notebook 03: Fine-Tuning Embeddings

## Why This Matters
General-purpose sentence-transformers like `all-MiniLM-L6-v2` are trained on broad internet text. They work well for general similarity but often underperform on specialized domains — legal documents, medical literature, code, or customer support conversations use vocabulary and concepts that shift the meaning of words in ways the general model hasn't seen.

Fine-tuning adapts the embedding model to your domain using contrastive learning on labeled or mined pairs. Done right, it can close the gap to a domain-specific model while starting from a strong general foundation. This is the standard approach in production retrieval systems today.

## Structure
1. When to fine-tune vs use off-the-shelf (narrative)
2. Contrastive loss functions — what's available and when to use each
3. Building a training dataset — positive/negative pair mining
4. Fine-tuning with sentence-transformers
5. Evaluating improvement on domain-specific STS
6. Hard negative mining — the key to training quality
7. Bridge to contrastive learning foundations

## What You'll Learn
- When domain fine-tuning meaningfully improves retrieval quality
- The difference between MultipleNegativesRankingLoss, CosineSimilarityLoss, and TripletLoss
- How to mine hard negatives from an unlabeled corpus
- How to evaluate embedding quality before and after fine-tuning


## Background

### When general models fail

A model trained on Wikipedia and Common Crawl has never seen "systolic dysfunction" used in proximity to "ejection fraction." It hasn't learned that "consideration" in a contract means something different than in casual conversation. The geometry of its embedding space encodes general English, not your domain.

Symptoms of a poorly matched embedding model:
- Retrieval recalls semantically wrong documents with high scores
- Documents you know are similar rank far apart
- Domain-specific terms cluster with unrelated general words

### The contrastive fine-tuning recipe

```
1. Collect (query, positive_doc) pairs from your domain
2. Mine hard negatives — documents that look relevant but aren't
3. Fine-tune with a ranking loss: pull positives close, push negatives away
4. Evaluate on a held-out domain STS or retrieval benchmark
```

The key variable is **negative quality**. Random negatives are easy — the model already handles those. Hard negatives — documents that share vocabulary with the query but answer a different question — force the model to learn fine-grained distinctions.

### Loss functions

**MultipleNegativesRankingLoss (MNRL):** Takes (query, positive) pairs; treats all other positives in the batch as negatives. Efficient because you don't need explicit negatives — the batch provides them. Most commonly used.

**CosineSimilarityLoss:** Takes (sentence_a, sentence_b, score) triples where score ∈ [0,1]. Good when you have human similarity ratings.

**TripletLoss:** Takes (anchor, positive, negative) triples. Requires explicit negatives; works well with hard negative mining.

**CachedMultipleNegativesRankingLoss:** Like MNRL but uses gradient caching to support very large batch sizes — important for quality since more in-batch negatives = harder training signal.


In [ ]:
# %pip install -q sentence-transformers torch datasets scikit-learn numpy

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from torch.utils.data import DataLoader
from scipy.stats import spearmanr

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

base_model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(base_model_name, device=device)
print(f"Loaded: {base_model_name}")

## 1. Baseline: General Model on Domain Data

In [ ]:
# Simulate a domain-specific corpus: ML systems and inference terminology
# In production this would be your actual document collection
domain_pairs = [
    # (query, positive_document, similarity_score)
    ("how does the kv cache work", "The KV cache stores key and value tensors from previous tokens to avoid recomputing attention on every step.", 1.0),
    ("what is speculative decoding", "Speculative decoding uses a small draft model to propose tokens, then verifies them in parallel with the full model.", 1.0),
    ("explain flash attention memory savings", "Flash attention tiles the attention computation to avoid materializing the full N×N attention matrix, reducing memory from O(N²) to O(N).", 1.0),
    ("how does continuous batching improve throughput", "Continuous batching allows new requests to join an in-flight batch as soon as a sequence finishes, maximizing GPU utilization.", 1.0),
    ("what is quantization in model serving", "Quantization reduces model weights from FP32 or BF16 to INT8 or INT4, reducing memory footprint and increasing inference speed.", 1.0),
    ("how does paged attention handle memory fragmentation", "PagedAttention manages KV cache memory using virtual memory paging, allocating non-contiguous blocks to avoid fragmentation.", 1.0),
    ("what is the purpose of rotary position embeddings", "RoPE encodes position by rotating query and key vectors in the complex plane, enabling better length generalization than learned absolute positions.", 1.0),
    ("explain prefix caching", "Prefix caching reuses the KV cache for shared prompt prefixes across requests, reducing compute for system prompts that appear in every call.", 1.0),
    # Hard negatives — share vocabulary but answer different questions
    ("how does the kv cache work", "Flash attention tiles the attention computation to reduce memory usage during the forward pass.", 0.1),
    ("what is speculative decoding", "Continuous batching allows new requests to join an in-flight batch dynamically.", 0.1),
    ("explain flash attention memory savings", "The KV cache avoids recomputing attention by storing previous key-value pairs.", 0.1),
]

# Evaluate baseline model
queries  = [p[0] for p in domain_pairs[:8]]
docs     = [p[1] for p in domain_pairs[:8]]
scores   = [p[2] for p in domain_pairs[:8]]

q_embs = model.encode(queries)
d_embs = model.encode(docs)

cosines = [float(np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d)))
           for q, d in zip(q_embs, d_embs)]

print("Baseline model — domain pair cosine similarities:")
for (q, _, _), cos in zip(domain_pairs[:8], cosines):
    print(f"  {cos:.3f}  {q[:55]}")
print(f"\nMean cosine (positives): {np.mean(cosines):.3f}")

## 2. Building a Training Dataset

In [ ]:
from sentence_transformers import InputExample

# Positive pairs for MultipleNegativesRankingLoss
# MNRL only needs (query, positive) — batch negatives are used automatically
train_examples_mnrl = [
    InputExample(texts=["how does the kv cache work",
                        "The KV cache stores key and value tensors to avoid recomputing attention on each step."]),
    InputExample(texts=["what is speculative decoding",
                        "Speculative decoding proposes tokens with a draft model then verifies in parallel with the full model."]),
    InputExample(texts=["explain flash attention memory savings",
                        "Flash attention tiles computation to avoid the full N×N matrix, reducing memory from O(N²) to O(N)."]),
    InputExample(texts=["how does continuous batching improve throughput",
                        "Continuous batching inserts new requests into ongoing batches as sequences finish, maximizing GPU utilization."]),
    InputExample(texts=["what is quantization in model serving",
                        "Quantization lowers weight precision from FP32 to INT8 or INT4, reducing memory and increasing throughput."]),
    InputExample(texts=["how does paged attention handle memory fragmentation",
                        "PagedAttention uses virtual memory paging to allocate non-contiguous KV cache blocks."]),
    InputExample(texts=["what is the purpose of rotary position embeddings",
                        "RoPE encodes position by rotating queries and keys in the complex plane for better length generalization."]),
    InputExample(texts=["explain prefix caching",
                        "Prefix caching reuses the KV cache for shared prompt prefixes, reducing compute on repeated system prompts."]),
    InputExample(texts=["what causes memory fragmentation in KV cache",
                        "Variable-length sequences cause internal and external fragmentation when KV cache is pre-allocated per sequence."]),
    InputExample(texts=["how does temperature affect sampling",
                        "Temperature scales logits before softmax — higher values flatten the distribution, increasing diversity."]),
    InputExample(texts=["what is top-p nucleus sampling",
                        "Top-p sampling truncates the vocabulary to the smallest set of tokens whose cumulative probability exceeds p."]),
    InputExample(texts=["explain tensor parallelism",
                        "Tensor parallelism splits weight matrices across GPUs, computing partial results in parallel and all-reducing."]),
]

print(f"Training examples: {len(train_examples_mnrl)}")
print("Format: (query, positive_document) pairs")
print("MNRL treats all other positives in the batch as negatives — no explicit negatives needed.")

## 3. Fine-Tuning with MultipleNegativesRankingLoss

In [ ]:
from sentence_transformers import losses

# Fresh copy of the base model to fine-tune
ft_model = SentenceTransformer(base_model_name, device=device)

train_dataloader = DataLoader(train_examples_mnrl, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

print("Loss: MultipleNegativesRankingLoss")
print("  — treats all (query, other_positive) pairs in the batch as negatives")
print("  — effective batch size = batch_size² negative pairs")
print(f"  — with batch_size=8: {8} positives, up to {8*7} in-batch negatives")
print()

# Fine-tune — short for demo; production would use 1-10 epochs on thousands of pairs
ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    warmup_steps=5,
    show_progress_bar=True,
    output_path="/tmp/ft_embedding_model",
)

print("\nFine-tuning complete.")

## 4. Evaluating Improvement

In [ ]:
# Re-evaluate on the same domain pairs
ft_q_embs = ft_model.encode(queries)
ft_d_embs = ft_model.encode(docs)

ft_cosines = [float(np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d)))
              for q, d in zip(ft_q_embs, ft_d_embs)]

print(f"{'Query':<55} {'Base':>6} {'Fine-tuned':>11}")
print("-" * 74)
for (q, _, _), base, ft in zip(domain_pairs[:8], cosines, ft_cosines):
    print(f"  {q[:53]:<53} {base:>6.3f} {ft:>11.3f}")

print()
print(f"Mean cosine — base model:       {np.mean(cosines):.3f}")
print(f"Mean cosine — fine-tuned model: {np.mean(ft_cosines):.3f}")
print()
print("Note: small training set and short training — production gains are larger.")
print("Key signal: fine-tuned scores should be higher and more discriminative.")

## 5. Hard Negative Mining

In [ ]:
# Hard negatives: documents that share vocabulary with the query but don't answer it
# These are the training examples that produce the biggest quality gains

# Strategy 1: BM25 retrieval — high lexical overlap, low semantic match
# Strategy 2: Embedding retrieval from base model — near misses in embedding space
# Strategy 3: Cross-encoder re-ranking — use a cross-encoder to score retrieved docs,
#              use low-scoring but high-embedding-similarity docs as hard negatives

# Demonstrate hard negative mining via embedding similarity
corpus = [p[1] for p in domain_pairs[:8]]
corpus_embs = model.encode(corpus)  # base model

print("Hard negative mining: for each query, find nearest corpus docs that are NOT the positive")
print()
for i, (q, pos, _) in enumerate(domain_pairs[:4]):
    q_emb = model.encode([q])[0]
    sims = [(j, float(np.dot(q_emb, d) / (np.linalg.norm(q_emb) * np.linalg.norm(d))))
            for j, d in enumerate(corpus_embs) if j != i]
    sims.sort(key=lambda x: -x[1])
    hard_neg_idx, hard_neg_sim = sims[0]  # highest sim that isn't the positive
    print(f"Query:     {q}")
    print(f"Positive:  {pos[:70]}")
    print(f"Hard neg:  {corpus[hard_neg_idx][:70]}  (sim={hard_neg_sim:.3f})")
    print()

print("These hard negatives, added to training, push the model to learn finer distinctions.")

## 6. Loss Function Cheat Sheet

```python
# 1. MultipleNegativesRankingLoss — most common, only needs (query, positive) pairs
loss = losses.MultipleNegativesRankingLoss(model)
# InputExample(texts=[query, positive])

# 2. CosineSimilarityLoss — for labeled similarity scores
loss = losses.CosineSimilarityLoss(model)  
# InputExample(texts=[sent_a, sent_b], label=0.85)

# 3. TripletLoss — explicit (anchor, positive, negative) triples
loss = losses.TripletLoss(model)
# InputExample(texts=[anchor, positive, negative])

# 4. CachedMultipleNegativesRankingLoss — MNRL with gradient cache for large batches
loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)
# Same format as MNRL; enables effective batch sizes of 512-2048

# Rule of thumb:
# - Have (query, doc) pairs only → MNRL
# - Have similarity scores → CosineSimilarityLoss  
# - Can mine explicit hard negatives → TripletLoss or MNRL with negatives
# - Need quality > speed, have GPU → CachedMNRL with large batch
```


## 7. Bridge to Contrastive Learning Foundations

Fine-tuning sentence-transformers *is* contrastive learning — you're pulling similar pairs together and pushing dissimilar pairs apart in embedding space using a learned distance metric.

The key abstractions that generalize beyond text:

- **Positive pairs** — in MNRL these are (query, relevant doc); in SimCLR these are two augmented views of the same image
- **In-batch negatives** — MNRL's batch trick is identical to SimCLR's: treat all other examples in the batch as negatives
- **Temperature** — both MNRL and NT-Xent scale similarity scores by a temperature before the softmax
- **Hard negatives** — the most impactful training improvement in both text and vision contrastive learning

Notebook 04 builds SimCLR from scratch, where these same ideas are implemented for images. The objective is structurally identical to what you just did here — different modality, same geometry.


## Summary

| Concept | Key Point |
|---------|-----------|
| When to fine-tune | Domain vocabulary shift, retrieval quality below threshold, labeled pairs available |
| MNRL | Best default: only needs (query, positive), uses in-batch negatives automatically |
| Hard negatives | Single biggest quality lever — semantically near but informationally wrong |
| Evaluation | Domain STS Spearman + retrieval recall@k on held-out queries |
| Production recipe | Base model → MNRL on domain pairs → hard negative mining → second-stage fine-tune |

**Next:** Notebook 04 — Contrastive Learning Foundations. SimCLR from scratch: augmentation strategy, NT-Xent loss, projection head, training loop. The same objective you just used for text, applied to images.
